# ☀️ Energybae — Solar Load Calculator
### Electricity Bill → Solar Recommendation → Excel Report

**AI Intern Practical Task | Energybae, Pimpri, Pune**

---

**How to use this notebook:**
1. Run **Step 1** once to install packages *(only needed the first time in a Colab session)*
2. Run **Step 2** to enter your OpenAI API key
3. Run **Step 3** — all the core functions are defined here *(run once)*
4. Run **Step 4** — upload your electricity bill (PDF, JPG, or PNG) and process it
5. Run **Step 5** — view the extracted data + solar recommendation, then download the Excel file

**Supported bill types:** MSEDCL, BESCOM, TATA Power, CESC, Adani Electricity, and other Indian utilities

---

## Step 1 — Install required packages
Run this cell once per Colab session.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai>=1.0', 'pdfplumber', 'Pillow', 'openpyxl'
])
print('✅ All packages installed successfully.')

## Step 2 — Set your OpenAI API key

**Option A (recommended for Colab):** Store your key as a Colab Secret:
- Click the 🔑 **Secrets** icon in the left sidebar
- Add a secret named `OPENAI_API_KEY` with your key value
- The cell below will read it automatically

**Option B:** Paste your key directly (uncomment the line, but don't share this notebook after doing so)

In [ ]:
import os

# ── Option B: paste your key here (remove before sharing!) ─────────────────
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'  # ← paste here

# ── Try Colab Secrets first, then fall back to environment variable ─────────
try:
    from google.colab import userdata
    _key = userdata.get('OPENAI_API_KEY')
    if _key:
        os.environ['OPENAI_API_KEY'] = _key
        print('✅ API key loaded from Colab Secrets.')
    else:
        raise KeyError('not found in secrets')
except Exception:
    if os.environ.get('OPENAI_API_KEY'):
        print('✅ API key found in environment.')
    else:
        print('⚠️  WARNING: No API key found.')
        print('   → Add it via Colab Secrets (🔑 in the left sidebar) or')
        print('     uncomment the os.environ line above and paste your key.')

## Step 3 — Define all helper functions
Run this cell once. It defines the AI extractor, solar calculator, and Excel generator.

In [ ]:
"""
═══════════════════════════════════════════════════════════════
  ENERGYBAE — SOLAR LOAD CALCULATOR  (self-contained module)
  All logic is defined inline so this notebook works in
  Google Colab without any extra files.
═══════════════════════════════════════════════════════════════
"""

# ─── Imports ────────────────────────────────────────────────────────────────
import os, io, json, base64, re, math
import openai
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ════════════════════════════════════════════════════════════════════════════
#  PART 1 — AI-POWERED BILL EXTRACTOR
# ════════════════════════════════════════════════════════════════════════════

EXTRACTION_PROMPT = """You are an expert at reading Indian electricity bills from utilities like MSEDCL, BESCOM, TATA Power, CESC, Adani Electricity, etc.

Extract ONLY these fields from the bill:
- Consumer Name
- Consumer Number / Account Number
- Billing Month (format: "Month YYYY", e.g., "March 2025")
- Units Consumed in kWh — total units this billing period (look for "Units consumed", "Total units", "Net units")
- Sanctioned Load in kW — contracted/maximum allowed load (look for "Sanctioned load", "Connected load", "Contract demand")
- Tariff Category — tariff slab or category code (e.g., LT-I, LT-II, HT, Domestic, Commercial, Industrial)
- Total Bill Amount in INR — the final payable amount
- Meter Number — meter serial number if visible
- Distribution Company — name of the electricity provider (e.g., MSEDCL, BESCOM, TATA Power)

Rules:
- Return ONLY a valid JSON object, no explanation, no markdown
- Use null for fields you cannot find
- Numbers must be numeric (not strings)
- If "units consumed" appears multiple times, use the NET or TOTAL value

JSON format:
{
  "consumer_name": "",
  "consumer_number": "",
  "billing_month": "",
  "units_consumed": 0,
  "sanctioned_load": 0,
  "tariff_category": "",
  "total_bill_amount": 0,
  "meter_number": null,
  "distribution_company": null
}"""


def _get_openai_client():
    api_key = os.environ.get('OPENAI_API_KEY')
    if not api_key:
        raise ValueError(
            'No OpenAI API key found. Run Step 2 to configure your key.'
        )
    return openai.OpenAI(api_key=api_key)


def _parse_ai_response(content: str) -> dict:
    """Extract JSON from AI response, tolerating markdown code blocks."""
    try:
        return json.loads(content.strip())
    except json.JSONDecodeError:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', content, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r'\{[\s\S]*\}', content)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    raise ValueError(f'Could not parse JSON from AI response:\n{content}')


def _sanitise(raw: dict) -> dict:
    def to_float(val, default=0.0):
        try:
            return float(str(val).replace(',', '').strip())
        except (TypeError, ValueError):
            return default
    return {
        'consumer_name':      str(raw.get('consumer_name') or 'Unknown'),
        'consumer_number':    str(raw.get('consumer_number') or 'Unknown'),
        'billing_month':      str(raw.get('billing_month') or 'Unknown'),
        'units_consumed':     to_float(raw.get('units_consumed')),
        'sanctioned_load':    to_float(raw.get('sanctioned_load')),
        'tariff_category':    str(raw.get('tariff_category') or 'Unknown'),
        'total_bill_amount':  to_float(raw.get('total_bill_amount')),
        'meter_number':       str(raw['meter_number']) if raw.get('meter_number') else None,
        'distribution_company': str(raw['distribution_company']) if raw.get('distribution_company') else None,
    }


def extract_from_pdf(pdf_bytes: bytes) -> dict:
    """Extract bill data from a text-based PDF."""
    import pdfplumber
    raw_text = ''
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                raw_text += text + '\n'
    if not raw_text.strip() or len(raw_text.strip()) < 30:
        raise ValueError(
            'The PDF appears to be scanned (no extractable text). '
            'Please convert it to a JPG/PNG image and upload that instead.'
        )
    client = _get_openai_client()
    response = client.chat.completions.create(
        model='gpt-4o',
        max_tokens=1024,
        messages=[{'role': 'user', 'content': f'{EXTRACTION_PROMPT}\n\nElectricity bill text:\n{raw_text}'}],
    )
    return _sanitise(_parse_ai_response(response.choices[0].message.content or '{}'))


def extract_from_image(image_bytes: bytes, mime_type: str = 'image/jpeg') -> dict:
    """Extract bill data from an image using GPT-4o vision."""
    b64 = base64.b64encode(image_bytes).decode('utf-8')
    data_url = f'data:{mime_type};base64,{b64}'
    client = _get_openai_client()
    response = client.chat.completions.create(
        model='gpt-4o',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image_url', 'image_url': {'url': data_url}},
                {'type': 'text', 'text': EXTRACTION_PROMPT},
            ],
        }],
    )
    return _sanitise(_parse_ai_response(response.choices[0].message.content or '{}'))


def extract_bill_data(file_bytes: bytes, filename: str) -> dict:
    """
    Main entry point. Auto-detects file type and extracts bill data.
    Supports PDF (text-based), JPG, and PNG.
    """
    lower = filename.lower()
    if lower.endswith('.pdf'):
        return extract_from_pdf(file_bytes)
    elif lower.endswith(('.jpg', '.jpeg')):
        return extract_from_image(file_bytes, 'image/jpeg')
    elif lower.endswith('.png'):
        return extract_from_image(file_bytes, 'image/png')
    else:
        raise ValueError(f'Unsupported file type: {filename}. Use PDF, JPG, or PNG.')


# ════════════════════════════════════════════════════════════════════════════
#  PART 2 — SOLAR CALCULATOR
# ════════════════════════════════════════════════════════════════════════════

PEAK_SUN_HOURS        = 4.5      # Average peak sun hours/day in India
SOLAR_GEN_FACTOR      = 4.0      # kWh generated per kW per day (conservative)
SYSTEM_COST_PER_KWP   = 60_000   # INR per kWp installed (approximate)
GRID_EMISSION_FACTOR  = 0.82     # kg CO₂ per kWh (India grid, CEA 2023)
SYSTEM_LIFE_YEARS     = 25


def calculate_solar_recommendation(bill_data: dict) -> dict:
    units   = bill_data.get('units_consumed', 0) or 0
    amount  = bill_data.get('total_bill_amount', 0) or 0
    s_load  = bill_data.get('sanctioned_load', 0) or 0

    daily_units       = units / 30
    size_by_consumption = daily_units / PEAK_SUN_HOURS
    size_by_load        = s_load * 0.8
    raw_size            = max(size_by_consumption, size_by_load)
    recommended_kw      = math.ceil(raw_size * 2) / 2 if raw_size > 0 else 1.0

    cost_per_unit    = (amount / units) if units > 0 else 7.0
    monthly_gen      = recommended_kw * SOLAR_GEN_FACTOR * 30
    covered_units    = min(monthly_gen, units)
    monthly_savings  = round(covered_units * cost_per_unit)
    annual_savings   = monthly_savings * 12
    system_cost      = recommended_kw * SYSTEM_COST_PER_KWP
    payback_years    = round(system_cost / annual_savings, 1) if annual_savings > 0 else 0.0
    annual_gen       = monthly_gen * 12
    co2_reduction    = round(annual_gen * GRID_EMISSION_FACTOR)

    return {
        'recommended_system_size_kw':  recommended_kw,
        'estimated_monthly_savings':   monthly_savings,
        'estimated_annual_savings':    annual_savings,
        'payback_period_years':        payback_years,
        'co2_reduction_kg_per_year':   co2_reduction,
        'system_cost_inr':             round(system_cost),
        'lifetime_savings_inr':        annual_savings * SYSTEM_LIFE_YEARS,
        'net_benefit_inr':             (annual_savings * SYSTEM_LIFE_YEARS) - system_cost,
        'daily_units':                 round(daily_units, 2),
        'monthly_generation_kwh':      round(monthly_gen, 1),
        'cost_per_unit':               round(cost_per_unit, 2),
    }


# ════════════════════════════════════════════════════════════════════════════
#  PART 3 — EXCEL REPORT GENERATOR
# ════════════════════════════════════════════════════════════════════════════

GREEN_DARK = '2C7A2C'
BLUE_DARK  = '1A5276'
GREY_LIGHT = 'F0F0F0'
WHITE      = 'FFFFFF'
GREY_MED   = '777777'


def _thin_border():
    thin = Side(style='thin', color='CCCCCC')
    return Border(top=thin, bottom=thin, left=thin, right=thin)

def _fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def _section_header(ws, row, title):
    ws.merge_cells(f'A{row}:F{row}')
    c = ws[f'A{row}']
    c.value = title
    c.font = Font(bold=True, size=12, color=WHITE)
    c.fill = _fill(BLUE_DARK)
    c.alignment = Alignment(horizontal='left', vertical='middle', indent=1)
    ws.row_dimensions[row].height = 24

def _data_row(ws, row, label, value, unit=''):
    lc = ws.cell(row=row, column=1, value=label)
    lc.fill = _fill(GREY_LIGHT)
    lc.border = _thin_border()
    ws.merge_cells(f'B{row}:D{row}')
    vc = ws.cell(row=row, column=2, value=value)
    vc.border = _thin_border()
    if unit:
        ws.merge_cells(f'E{row}:F{row}')
        uc = ws.cell(row=row, column=5, value=unit)
        uc.font = Font(color=GREY_MED, italic=True)
    ws.row_dimensions[row].height = 22

def _fmt_inr(amount):
    try:
        n = int(round(amount))
        s = str(n)
        if len(s) <= 3:
            return f'₹ {s}'
        result = s[-3:]
        s = s[:-3]
        while s:
            result = s[-2:] + ',' + result
            s = s[:-2]
        return f'₹ {result.lstrip(",")}'
    except Exception:
        return f'₹ {amount}'


def generate_excel(bill_data: dict, solar: dict) -> bytes:
    """Generate a formatted Excel workbook and return raw bytes."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Solar Load Calculator'

    for i, w in enumerate([30, 22, 16, 16, 16, 16], 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title
    ws.merge_cells('A1:F1')
    t = ws['A1']
    t.value = 'ENERGYBAE — Solar Load Calculator'
    t.font = Font(bold=True, size=16, color=WHITE)
    t.fill = _fill(GREEN_DARK)
    t.alignment = Alignment(horizontal='center', vertical='middle')
    ws.row_dimensions[1].height = 36

    ws.merge_cells('A2:F2')
    s = ws['A2']
    s.value = 'Electricity Bill Analysis & Solar System Recommendation'
    s.font = Font(italic=True, size=11, color='555555')
    s.alignment = Alignment(horizontal='center')
    ws.row_dimensions[2].height = 20

    # Section 1
    _section_header(ws, 4,  'SECTION 1 — Customer Information')
    _data_row(ws, 5,  'Consumer Name',        bill_data.get('consumer_name', 'N/A'))
    _data_row(ws, 6,  'Consumer Number',       bill_data.get('consumer_number', 'N/A'))
    _data_row(ws, 7,  'Meter Number',          bill_data.get('meter_number') or 'N/A')
    _data_row(ws, 8,  'Distribution Company',  bill_data.get('distribution_company') or 'N/A')
    _data_row(ws, 9,  'Billing Month',         bill_data.get('billing_month', 'N/A'))
    _data_row(ws, 10, 'Tariff Category',       bill_data.get('tariff_category', 'N/A'))

    # Section 2
    _section_header(ws, 12, 'SECTION 2 — Electricity Usage')
    _data_row(ws, 13, 'Units Consumed',            bill_data.get('units_consumed', 0),                'kWh / month')
    _data_row(ws, 14, 'Sanctioned Load',            bill_data.get('sanctioned_load', 0),              'kW')
    _data_row(ws, 15, 'Total Bill Amount',          _fmt_inr(bill_data.get('total_bill_amount', 0)))
    _data_row(ws, 16, 'Cost per Unit',              f"₹ {solar.get('cost_per_unit', 0):.2f}",         '₹ / kWh')
    _data_row(ws, 17, 'Average Daily Consumption',  f"{solar.get('daily_units', 0):.2f}",             'kWh / day')

    # Section 3
    _section_header(ws, 19, 'SECTION 3 — Solar System Recommendation')
    for row, label, value, unit in [
        (20, 'Recommended System Size',   solar.get('recommended_system_size_kw', 0), 'kWp'),
        (21, 'Estimated Monthly Savings', _fmt_inr(solar.get('estimated_monthly_savings', 0)), ''),
        (22, 'Estimated Annual Savings',  _fmt_inr(solar.get('estimated_annual_savings', 0)), ''),
        (23, 'Estimated Payback Period',  solar.get('payback_period_years', 0), 'years'),
        (24, 'CO₂ Reduction',             f"{solar.get('co2_reduction_kg_per_year', 0):,}", 'kg CO₂ / year'),
    ]:
        _data_row(ws, row, label, value, unit)
        if row in (20, 21, 22):
            ws.cell(row=row, column=1).font = Font(bold=True)
            ws.cell(row=row, column=2).font = Font(bold=True)

    # Section 4
    _section_header(ws, 26, 'SECTION 4 — Financial Summary (25-year projection)')
    _data_row(ws, 27, 'Estimated System Cost',    _fmt_inr(solar.get('system_cost_inr', 0)),    '(approx. ₹60,000/kWp installed)')
    _data_row(ws, 28, '25-Year Savings',           _fmt_inr(solar.get('lifetime_savings_inr', 0)))
    _data_row(ws, 29, 'Net Benefit (25yr − Cost)', _fmt_inr(solar.get('net_benefit_inr', 0)))

    # Footer
    ws.merge_cells('A31:F31')
    f1 = ws['A31']
    f1.value = 'Generated by Energybae Solar Load Calculator | www.energybae.in | energybae.co@gmail.com'
    f1.font = Font(italic=True, size=9, color='888888')
    f1.alignment = Alignment(horizontal='center')

    ws.merge_cells('A32:F32')
    f2 = ws['A32']
    f2.value = '* Estimates based on 4.5 peak sun hours/day (India avg.) and ₹60,000/kWp installation cost. Actual results may vary.'
    f2.font = Font(italic=True, size=8, color='AAAAAA')
    f2.alignment = Alignment(horizontal='center')

    buf = io.BytesIO()
    wb.save(buf)
    return buf.getvalue()


print('✅ All functions defined and ready.')
print('   extract_bill_data()           — AI extraction from PDF / image')
print('   calculate_solar_recommendation() — Solar sizing & savings')
print('   generate_excel()              — Formatted Excel report')

## Step 4 — Upload an electricity bill and process it

A file picker will appear. Select your electricity bill (PDF, JPG, or PNG).
AI extraction takes about 10–20 seconds.

In [ ]:
from google.colab import files as colab_files

# ── Upload ───────────────────────────────────────────────────────────────────
print('📁 A file picker will appear below. Select your electricity bill.')
uploaded = colab_files.upload()

if not uploaded:
    print('❌ No file selected. Please run this cell again and choose a file.')
else:
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]
    print(f'\n📄 File: {filename}  ({len(file_bytes)/1024:.1f} KB)')
    print()

    # ── Extract ──────────────────────────────────────────────────────────────
    print('⏳ Step 1/3 — Sending bill to GPT-4o for extraction (10-20 seconds)...')
    try:
        bill_data = extract_bill_data(file_bytes, filename)
        print('✅ Bill data extracted.')
    except Exception as e:
        print(f'❌ Extraction failed: {e}')
        raise

    # ── Calculate ────────────────────────────────────────────────────────────
    print('⏳ Step 2/3 — Calculating solar recommendation...')
    solar = calculate_solar_recommendation(bill_data)
    print('✅ Solar recommendation calculated.')

    # ── Excel ────────────────────────────────────────────────────────────────
    print('⏳ Step 3/3 — Generating Excel report...')
    try:
        excel_bytes = generate_excel(bill_data, solar)
        print('✅ Excel report generated.')
    except Exception as e:
        print(f'❌ Excel generation failed: {e}')
        raise

    print()
    print('🎉 Done! Run Step 5 below to view the results and download the Excel file.')

## Step 5 — View results and download the Excel report

In [ ]:
from google.colab import files as colab_files

try:
    bill_data
    solar
    excel_bytes
except NameError:
    print('❌ No results yet — please run Step 4 first.')
    raise SystemExit()

# ── Customer Info ─────────────────────────────────────────────────────────────
print('=' * 60)
print('  CUSTOMER INFORMATION')
print('=' * 60)
print(f"  Consumer Name      : {bill_data['consumer_name']}")
print(f"  Consumer Number    : {bill_data['consumer_number']}")
print(f"  Meter Number       : {bill_data.get('meter_number') or 'N/A'}")
print(f"  Distribution Co.   : {bill_data.get('distribution_company') or 'N/A'}")
print(f"  Billing Month      : {bill_data['billing_month']}")
print(f"  Tariff Category    : {bill_data['tariff_category']}")

# ── Electricity Usage ─────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  ELECTRICITY USAGE')
print('=' * 60)
print(f"  Units Consumed     : {bill_data['units_consumed']:,.0f} kWh")
print(f"  Sanctioned Load    : {bill_data['sanctioned_load']} kW")
print(f"  Total Bill Amount  : Rs. {bill_data['total_bill_amount']:,.0f}")
print(f"  Cost per Unit      : Rs. {solar['cost_per_unit']:.2f} / kWh")
print(f"  Avg Daily Usage    : {solar['daily_units']:.2f} kWh / day")

# ── Solar Recommendation ──────────────────────────────────────────────────────
print()
print('=' * 60)
print('  SOLAR SYSTEM RECOMMENDATION')
print('=' * 60)
print(f"  System Size        : {solar['recommended_system_size_kw']} kWp")
print(f"  Monthly Savings    : Rs. {solar['estimated_monthly_savings']:,}")
print(f"  Annual Savings     : Rs. {solar['estimated_annual_savings']:,}")
print(f"  System Cost (est.) : Rs. {solar['system_cost_inr']:,}")
print(f"  Payback Period     : {solar['payback_period_years']} years")
print(f"  CO2 Reduction      : {solar['co2_reduction_kg_per_year']:,} kg / year")
print(f"  25-yr Net Benefit  : Rs. {solar['net_benefit_inr']:,}")

# ── Download Excel ────────────────────────────────────────────────────────────
print()
print('=' * 60)
excel_filename = f"solar_load_{bill_data['consumer_number'].replace(' ', '_')}.xlsx"
with open(excel_filename, 'wb') as f:
    f.write(excel_bytes)
print(f'📥 Downloading Excel report: {excel_filename}')
colab_files.download(excel_filename)

---
## How it works

| Step | What happens |
|------|--------------|
| File reading | **PDF:** `pdfplumber` extracts text. **Image:** base64-encoded and sent to GPT-4o vision. |
| AI extraction | A structured prompt asks GPT-4o to return a JSON with 9 fields (consumer name, units, load, amount, etc.) |
| Solar sizing | `max(daily_units ÷ 4.5, sanctioned_load × 0.8)` rounded up to nearest 0.5 kWp |
| Savings | Monthly generation × cost-per-unit from the bill |
| Payback | System cost ÷ annual savings |
| CO₂ | Annual generation × 0.82 kg/kWh (India grid emission factor, CEA 2023) |
| Excel | `openpyxl` creates a formatted 4-section workbook with colour-coded headers |

---
*Energybae — Empowering People with Renewable Energy Solutions*  
*www.energybae.in | energybae.co@gmail.com | +91 9112233120*